In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
# from tensorflow import keras

peppers = pd.read_csv("../data/labels.csv")
df = pd.DataFrame(peppers)
# unsure if I want to use the line below, it may potentially cause issues down the line
# df = df.set_index("pepper_id")


In [ ]:
# # """ 
# # CNN
# # """

# # load the fashion MINST dataset
# fashion_mnist = tf.keras.datasets.fashion_mnist

# # splits data into trainging and testing
# (train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# # names of the various classes within the dataset
# class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
#                'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# # # normalizes pixel values from 0-255 down to 0-1, the image pixels are originally integers 
# # like 0 to 255. Dividing by 255 scales them to decimals between 0 and 1, which usually helps 
# # the model train better
# train_images = train_images / 255.0
# test_images = test_images / 255.0

# # creates the model
# model = tf.keras.Sequential([
#     # adds a channel dimension by reshaping each image from 2D to 3D so it matches what the CNN expects
#     tf.keras.layers.Reshape((28, 28, 1), input_shape=(28, 28)),
#     # creates a convolutional layer with 32 3x3 filters
#     tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
#     # reduces the amount of spatial information by keeping the strongest value in each 2x2 region
#     tf.keras.layers.MaxPooling2D((2, 2)),
#     # creates another convolutional layer with 64 3x3 filters
#     tf.keras.layers.Conv2D(64, (3, 3), activation='relu',padding='same'),
#     # reduces the amount of spatial information by keeping the strongest value in each 2x2 region
#     tf.keras.layers.MaxPooling2D((2, 2)),
#     # flattens the 3D feature maps into a 1D array so it can be passed to dense layers
#     tf.keras.layers.Flatten(),
#     # create a 128 neuron layer using relu activation function
#     tf.keras.layers.Dense(128, activation='relu'),
#     # randomly turns off 30% of the neurons during training to help prevent overfitting, 
#     # this is a form of regularization
#     tf.keras.layers.Dropout(0.3),
#     # creates the output layer with 10 neurons, one for each class
#     tf.keras.layers.Dense(10)
# ])

# # configures the model for training by setting the optimizer, loss function, and evaluation metric
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#               metrics=['accuracy'])

# # trains the model on the training data, using 10% of it for validation during training
# model.fit(train_images, train_labels, epochs=25,validation_split=0.1)

# # evaluates the trained model on the test data and returns the test loss and test accuracy
# test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)

# # prints test accuracy
# print('\nTest accuracy:', test_acc)


In [78]:
from pathlib import Path
import tensorflow as tf

project_dir = Path.cwd().parent
img_dir = project_dir / "img"
csv_dir = project_dir / "data"

print("project_dir:", project_dir)
print("img_dir exists:", img_dir.exists())
print("csv_dir exists:", csv_dir.exists())

img_height = 180
img_width = 180
batch_size = 32
seed = 123

train_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(180, 180),
    batch_size=32
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names
print("Class names:", class_names)

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

print("\nValidation accuracy:", val_acc)

project_dir: /Users/josephklinck/Desktop/ml-pyenv/jalapeno-predictor
img_dir exists: True
csv_dir exists: True
Found 197 files belonging to 3 classes.
Using 158 files for training.
Found 197 files belonging to 3 classes.
Using 39 files for validation.
Class names: ['Hot', 'Mild', 'medium']
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 278ms/step - accuracy: 0.3165 - loss: 3.9803 - val_accuracy: 0.3077 - val_loss: 1.1626
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.3354 - loss: 1.1229 - val_accuracy: 0.3077 - val_loss: 1.0689
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.4367 - loss: 1.0685 - val_accuracy: 0.2564 - val_loss: 1.1199
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.4051 - loss: 1.0773 - val_accuracy: 0.2564 - val_loss: 1.1573
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.4810 - loss: 1.0936 - val_accuracy: 0.5128 - val_loss: 0.9445
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.3734 - loss: 1.